# **Initialization**

In [1]:
"""Start"""

'Start'

In [ ]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import sys
import pulp
import vrplib
import re
import sys
import os
import gc
import contextlib
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
import time
from docplex.mp.model import Model

# --- 1. DEFINE PATH TO LIBRARY PARENT FOLDER ---
# Replace this with the ACTUAL path to the folder containing 'didp_ea_lib'
# IMPORTANT: Use r"..." string to handle Windows backslashes correctly
LIBRARY_PARENT_PATH = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\Evolutionary_algorithm"
# --- 2. ADD TO SYSTEM PATH ---
if LIBRARY_PARENT_PATH not in sys.path:
    sys.path.append(LIBRARY_PARENT_PATH)
print(f"Library path added: {LIBRARY_PARENT_PATH}")
# --- 3. TEST IMPORT ---
try:
    import evolutionary_algorithm_lib
    from evolutionary_algorithm_lib import *
    from evolutionary_algorithm_lib import (compile_chromosome_to_useable_function, 
                                            combining_modified_didppy_solver_with_chromosome)
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    
    print("✅ Success! 'evolutionary_algorithm_lib' is imported and ready.")
except ImportError as e:
    print(f"❌ Error: Could not import library. Check the path above.\nDetails: {e}")

Library path added: C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\3_Evolutionary_algorithm
✅ Success! 'evolutionary_algorithm_lib' is imported and ready.


# **Configuration & Data input**

In [ ]:
# --- CONFIGURATION ---
# Path to your n20 folder containing .txt files
DATA_DIR = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_TSPTW_dual_bounds_and_models\n20"
INPUT_CSV = "TSP_single_dual_bound_results.csv"
OUTPUT_CSV = "result_of_ea_dual_bounds.csv"
LOGS_DIR = "batch_logs"

if not os.path.exists(LOGS_DIR):
    os.makedirs(LOGS_DIR)

# --- GLOBAL VARIABLES (Initialize with Dummy Data) ---
# We create these so the functions in Cell 3 don't crash if checked early.
# These will be overwritten by the loop in Cell 4.
current_number_of_customers = 5
current_distance_list = [[0]*5 for _ in range(5)]

print("✅ Globals initialized.")

# --- BATCH UTILITIES ---
def get_processed_instances(csv_path, logs_dir):
    """
    Returns a set of instances that exist in BOTH the CSV summary and the logs folder.
    """
    # 1. Get instances from CSV
    csv_instances = set()
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if 'Instance' in df.columns:
                csv_instances = set(df['Instance'].unique())
        except:
            pass # CSV read failed, assume empty

    # 2. Get instances from Log Files
    log_instances = set()
    if os.path.exists(logs_dir):
        for filename in os.listdir(logs_dir):
            if filename.endswith("_log.txt"):
                # Extract "0.txt" from "0.txt_log.txt"
                instance_name = filename.replace("_log.txt", "")
                log_instances.add(instance_name)

    # 3. Return Intersection (Must be in BOTH to be considered "Done")
    return csv_instances.intersection(log_instances)

def append_result_to_csv(result_dict, csv_path):
    df = pd.DataFrame([result_dict])
    df.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)

print("✅ Configuration set.")

def read_tsp_cappart_format(file_path):
    with open(file_path, 'r') as f:
        values = f.read().split()
    iterator = iter(values)
    n = int(next(iterator))
    c = []
    for i in range(n):
        row = []
        for j in range(n):
            row.append(int(float(next(iterator))))
        c.append(row)
    return n, c

# Cell 3.5: Data Cleanup Utility

def clean_batch_data(csv_path, logs_dir):
    """
    Ensures consistency between the CSV summary and the Log files.
    1. Removes duplicate instances in CSV (keeps last).
    2. Removes CSV rows if the corresponding Log file is missing.
    3. Deletes Log files if the corresponding CSV row is missing.
    """
    print("🧹 Starting Data Cleanup...")
    
    # 1. Load CSV
    if not os.path.exists(csv_path):
        print("   -> CSV not found. Nothing to clean in CSV.")
        # If CSV missing but logs exist, we might want to clear logs, 
        # but usually better to leave them or delete manually to be safe.
        return 

    try:
        df = pd.read_csv(csv_path)
    except pd.errors.EmptyDataError:
        print("   -> CSV is empty.")
        return

    original_count = len(df)
    
    # 2. Deduplicate CSV (Keep the last run)
    df.drop_duplicates(subset=['Instance'], keep='last', inplace=True)
    dedup_count = len(df)
    if original_count > dedup_count:
        print(f"   -> Removed {original_count - dedup_count} duplicate rows from CSV.")

    # 3. Remove CSV rows without matching Log files
    valid_indices = []
    instances_in_csv = set()
    
    for index, row in df.iterrows():
        instance_name = row['Instance']
        expected_log = os.path.join(logs_dir, f"{instance_name}_log.txt")
        
        if os.path.exists(expected_log):
            valid_indices.append(index)
            instances_in_csv.add(instance_name)
        else:
            print(f"   -> Removing CSV row for '{instance_name}' (Log file missing).")
            
    # Filter dataframe to keep only valid rows
    df_clean = df.loc[valid_indices]
    
    # Save cleaned CSV
    df_clean.to_csv(csv_path, index=False)
    print(f"   -> CSV saved. Current number of row is: {len(df_clean)} (was {original_count}).")

    # 4. Remove Orphan Log files (Log exists, but not in CSV)
    if os.path.exists(logs_dir):
        files = os.listdir(logs_dir)
        for filename in files:
            if filename.endswith("_log.txt"):
                instance_from_log = filename.replace("_log.txt", "")
                
                if instance_from_log not in instances_in_csv:
                    file_path = os.path.join(logs_dir, filename)
                    try:
                        os.remove(file_path)
                        print(f"   -> Deleted orphan log: {filename} (Not in CSV).")
                    except OSError as e:
                        print(f"   -> Error deleting {filename}: {e}")

    print("✨ Data Cleanup Complete.\n")


✅ Globals initialized.
✅ Configuration set.


# **Model and dual bounds declaration**

In [6]:
def creation_of_didp_model_function():
    """
    Creates the CVRP DIDP model and returns it along with necessary metadata 
    for the heuristic functions.
    """
    n = current_number_of_customers
    c = current_distance_list
    
    # 2. Initialize Model
    # Note: Ensure float_cost matches your data. Your snippet used False (Int), 
    # so we explicitly cast distances to Int in the reader.
    model = m_dp.Model(maximize=False, float_cost=False)

    customer = model.add_object_type(number=n)

    # 3. State Variables
    # U: Unvisited set (excluding depot 0)
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))
    # i: Current location
    location = model.add_element_var(object_type=customer, target=0)

    # 4. Resource Tables
    travel_time = model.add_int_table(c)

    # 5. Transitions
    # Visit customer j
    for j in range(1, n):
        visit = m_dp.Transition(
            name="visit {}".format(j),
            cost=travel_time[location, j] + m_dp.IntExpr.state_cost(),
            preconditions=[unvisited.contains(j)],
            effects=[
                (unvisited, unvisited.remove(j)),
                (location, j),
            ],
        )
        model.add_transition(visit)

    # Return to depot
    # Note: Removed 'time' effect from your snippet as it wasn't defined in the variables
    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time[location, 0] + m_dp.IntExpr.state_cost(),
        effects=[
            (location, 0),
        ],
        preconditions=[unvisited.is_empty(), location != 0],
    )
    model.add_transition(return_to_depot)

    # 6. Base Case
    model.add_base_case([unvisited.is_empty(), location == 0])

    # 8. Create Bundle (Model + Metadata)
    # This metadata dict allows your heuristics (like MST or assignment) 
    # to access the raw matrix data later.
    metadata = {
        "num_nodes": n,
        "distance_matrix": c,
        "unvisited_var": unvisited,
        "location_var": location,
        # Add other keys if your dual bounds need them
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

In [5]:
def create_persistent_lp_relaxation_dual_bounds(metadata):
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    unvisited_set_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    dist_matrix = metadata['distance_matrix']
    
    # ==========================================
    # 1. INITIALIZATION (Runs Once)
    # ==========================================
    # Create the model instance only ONCE
    mdl = Model(name='Persistent_TSP_Relaxation')
    
    # Optimization Parameters for Speed
    mdl.parameters.threads = 1
    mdl.parameters.lpmethod = 1  # Primal Simplex is efficient for re-optimization
    mdl.log_output = False       # Silence output

    # --- Create Variables ---
    # x[i, j]: Flow variables (Continuous 0-1)
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                x[(i, j)] = mdl.continuous_var(lb=0, ub=1, name=f'x_{i}_{j}')

    # u[i]: MTZ potential variables
    u = {i: mdl.continuous_var(lb=0, ub=n_nodes, name=f'u_{i}') for i in range(n_nodes)}

    # --- Create Constraints (Store references to update them later) ---
    # We create constraints for ALL nodes initially.
    # We will toggle their RHS (Right Hand Side) between 1 and 0 dynamically.
    
    cons_out = {} # Constraint: Sum(x_ij) = RHS
    cons_in = {}  # Constraint: Sum(x_ji) = RHS
    
    for i in range(n_nodes):
        # Outgoing flow
        # sum(x[i, j] for all j) == RHS
        expr_out = mdl.sum(x[(i, j)] for j in range(n_nodes) if i != j)
        cons_out[i] = mdl.add_constraint(expr_out == 1, ctname=f'deg_out_{i}')
        
        # Incoming flow
        # sum(x[j, i] for all j) == RHS
        expr_in = mdl.sum(x[(j, i)] for j in range(n_nodes) if i != j)
        cons_in[i] = mdl.add_constraint(expr_in == 1, ctname=f'deg_in_{i}')

    # MTZ Constraints (Static - they rely on x and u values)
    # u[i] - u[j] + N * x[i,j] <= N - 1
    # We don't need to remove these; if x[i,j] is forced to 0, the constraint becomes loose (valid).
    for i in range(n_nodes):
        if i == 0: continue
        for j in range(n_nodes):
            if j == 0 or i == j: continue
            mdl.add_constraint(
                u[i] - u[j] + n_nodes * x[(i, j)] <= n_nodes - 1
            )

    # --- Static Objective ---
    obj_expr = mdl.sum(dist_matrix[i][j] * x[(i, j)] 
                    for i in range(n_nodes)
                    for j in range(n_nodes) if i != j)
    mdl.minimize(obj_expr)

    # ==========================================
    # 2. DYNAMIC HEURISTIC (Runs many times)
    # ==========================================
    def h_lp_relaxation(state):
        # A. Identify Active Nodes
        # The 'Active Subgraph' consists of: Current Node + Unvisited Nodes + Depot
        unvisited = state[unvisited_set_var]
        current_node = state[location_var]
        
        # Quick exit for solved state
        if not unvisited and current_node == 0: 
            return 0.0

        # Construct a fast lookup set for active nodes
        # 0 (Depot) is always part of the formulation in this relaxation
        active_set = set(unvisited)
        active_set.add(current_node)
        active_set.add(0) 

        # B. Update Model (The Optimization)
        # Instead of rebuilding, we just switch the "power" on/off for nodes
        
        for i in range(n_nodes):
            if i in active_set:
                # ACTIVE NODE: Must have degree 1 (Flow = 1)
                cons_out[i].rhs = 1
                cons_in[i].rhs = 1
                # Ensure u-variable is active (allowed to be > 0)
                u[i].ub = n_nodes
            else:
                # INACTIVE NODE: Must have degree 0 (Flow = 0)
                # Setting RHS to 0 forces all connected x_ij variables to 0
                # because x_ij >= 0. This effectively removes the node.
                cons_out[i].rhs = 0
                cons_in[i].rhs = 0
                # Fix u-variable to 0 to help solver
                u[i].ub = 0

        # C. Solve Re-optimized Model
        # cplex/docplex is smart enough to use the previous basis for speed
        sol = mdl.solve()
        
        if sol:
            return float(sol.objective_value)
        return 0.0

    return h_lp_relaxation

def dual_bound_expression_function(didp_bundle):
    " Returns a dictionary of heuristic functions (bounds) bound to the model data."
    
    model, metadata = didp_bundle
    
    # Extract metadata
    distance_list = metadata['distance_matrix']
    cost_matrix = np.array(distance_list) # Numpy version for calculations
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    num_nodes = metadata['num_nodes'] # Assumed available from creation function

    # --- Pre-computation for h_global_min_flow (Bound 2) ---
    # We calculate the global min outgoing and incoming edges for every node once.
    # This matches the 'min_to' and 'min_from' tables in the DIDP snippet.
    
    # masked_cost: diagonal is infinity to ignore self-loops
    masked_cost = cost_matrix.astype(float).copy()
    np.fill_diagonal(masked_cost, np.inf)
    
    # min_outgoing[i] = min cost to leave node i
    min_outgoing_arr = np.min(masked_cost, axis=1)
    
    # min_incoming[j] = min cost to enter node j
    min_incoming_arr = np.min(masked_cost, axis=0)

    # ==========================================
    # 1. Degree Average Bound (Local Subgraph) [NEW]
    # ==========================================
    def h_degree_average(state):
        U = state[unvisited_var]
        curr = state[location_var]
        
        # If no unvisited nodes and we are at depot (0), cost is 0
        if not U and curr == 0:
            return 0.0

        # Define active nodes for the path: Current -> [Unvisited] -> Depot (0)
        # We need to construct the submatrix for these specific nodes
        active_nodes = [curr] + sorted(list(U))
        if 0 not in active_nodes:
            active_nodes.append(0)
            
        # Extract submatrix
        sub_mat = cost_matrix[np.ix_(active_nodes, active_nodes)].astype(float)
        np.fill_diagonal(sub_mat, np.inf)

        # Calculate mins within this specific subgraph
        # axis=0 is min down columns (Incoming), axis=1 is min across rows (Outgoing)
        mins_in = np.min(sub_mat, axis=0) 
        mins_out = np.min(sub_mat, axis=1)
        
        # Logic for Path Constraints:
        # 1. Current Node (index 0 in active_nodes): Needs Outgoing, but NO Incoming
        # 2. Depot Node (index -1 in active_nodes): Needs Incoming, but NO Outgoing
        # 3. Intermediate (Unvisited): Need BOTH
        
        # Sum of valid Incoming edges (Everyone except Current)
        # Note: We must map the exclusion correctly. 
        # Since active_nodes[0] is 'curr', we exclude mins_in[0]
        sum_in = np.sum(mins_in[1:])
        
        # Sum of valid Outgoing edges (Everyone except Depot)
        # Since active_nodes[-1] is '0', we exclude mins_out[-1]
        sum_out = np.sum(mins_out[:-1])
        
        # Return average
        return float(0.5 * (sum_in + sum_out))

    # ==========================================
    # 2. Global Min Flow Bound (Max of Min-In/Out) [NEW]
    # ==========================================
    def h_global_min_flow(state):
        U = state[unvisited_var]
        curr = state[location_var]
        
        if not U and curr == 0:
            return 0.0

        # Bound A: Sum of minimum OUTGOING edges
        # We must leave 'curr' and every node in 'U'
        val_out = sum(min_outgoing_arr[u] for u in U)
        if curr != 0:
            val_out += min_outgoing_arr[curr]
            
        # Bound B: Sum of minimum INCOMING edges
        # We must enter '0' and every node in 'U'
        val_in = sum(min_incoming_arr[u] for u in U)
        if curr != 0: # If we aren't already at 0, we must eventually enter 0
            val_in += min_incoming_arr[0]
            
        # Return the tighter (maximum) of the two constraints
        return float(max(val_out, val_in))

    # ==========================================
    # 3. LP Relaxation Bound (On-the-fly)
    # ==========================================
    h_lp_relaxation = create_persistent_lp_relaxation_dual_bounds(metadata = metadata)

    # ==========================================
    # 4. MST Bound
    # ==========================================
    def h_mst(state):
        U = state[unvisited_var]
        if not U: return 0.0
        nodes = [0] + sorted(list(U))
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    # ==========================================
    # 5. 1-Tree Bound
    # ==========================================
    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        subset_nodes = sorted(list(U))
        
        depot_edges = sorted(cost_matrix[0, subset_nodes])
        e1 = depot_edges[0]
        e2 = depot_edges[1] if len(depot_edges) > 1 else 0.0
        
        if len(subset_nodes) > 1:
            sub_mat = cost_matrix[np.ix_(subset_nodes, subset_nodes)]
            mst_val = minimum_spanning_tree(sub_mat).sum()
        else:
            mst_val = 0.0 
        return float(mst_val + e1 + e2)

    # ==========================================
    # 6. Assignment Bound
    # ==========================================
    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        nodes = [0] + sorted(list(U))
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        # Fill diagonal with Infinity to forbid self-loops (i -> i)
        np.fill_diagonal(assign_mat, np.inf)
        # This finds the cheapest set of edges such that every row/col is used once
        row_ind, col_ind = linear_sum_assignment(assign_mat)
        return float(assign_mat[row_ind, col_ind].sum())

    # ==========================================
    # 7. Eigenvalue Bound
    # ==========================================
    def h_eigen(state):
        U = state[unvisited_var]
        nodes = [0] + sorted(list(U))
        N = len(nodes)
        if N < 2: return 0.0

        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N, 1))
        P = np.eye(N) - (one @ one.T) / N
        M = -P @ D_sub @ P
        M = (M + M.T) / 2
        try:
            eigvals = np.flip(eigh(M)[0])
        except np.linalg.LinAlgError:
            return 0.0
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N) for k in range(1, N)])

        phi = 0.0
        if N > 1:
            if N % 2 == 1:
                num_terms = (N - 1) // 2
                if 2 * num_terms <= len(eigvals) and num_terms <= len(coeffs):
                     phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_terms + 1))
                else:
                    print(f"⚠️ Not enough eigenvalues/coefficients for odd N formula (subset {U})")
            else:
                num_sum_terms = N // 2 - 1
                if 2 * num_sum_terms < len(eigvals) and num_sum_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_sum_terms + 1))
                    if N-2 < len(eigvals):
                        phi += 2 * eigvals[N - 2]
                elif N > 1 and N-2 < len(eigvals):
                    phi = 2 * eigvals[N - 2]
                else:
                    print(f"⚠️ Not enough eigenvalues/coefficients for odd N formula (subset {U})")
        return float(phi)
    
    # Return valid registry
    dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    return dual_bound_dict

dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())
display(dual_bound_functions_registry)

{'h_degree_average': <function __main__.dual_bound_expression_function.<locals>.h_degree_average(state)>,
 'h_global_min_flow': <function __main__.dual_bound_expression_function.<locals>.h_global_min_flow(state)>,
 'h_lp_relaxation': <function __main__.create_persistent_lp_relaxation_dual_bounds.<locals>.h_lp_relaxation(state)>,
 'h_mst': <function __main__.dual_bound_expression_function.<locals>.h_mst(state)>,
 'h_1tree': <function __main__.dual_bound_expression_function.<locals>.h_1tree(state)>,
 'h_assignment': <function __main__.dual_bound_expression_function.<locals>.h_assignment(state)>,
 'h_eigen': <function __main__.dual_bound_expression_function.<locals>.h_eigen(state)>}

# **Hyperparamters setting**

In [7]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 2        # Size of the population in each generation
GENERATIONS = 5           # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.02
# ==========================================
# 2. OPERATOR PARAMETERS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 
# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 10               # Maximum depth of the initial trees
# Probability of selecting the best individual in the  tournament selection
# Tournament size for parent selection
tournament_size=random.randint(2, 10)
tournament_probability=0.8
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length)  # Randomly chosen between 1 and 3
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5
# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9
# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5
# ==========================================
# 4. OTHER PARAMETERS
# ==========================================
# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
# OPTIMAL_COST_REFERENCE= 400
# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 5 #seconds

# **Execution**

In [8]:
# --- EXECUTE CLEANUP ---
# Run this right before your main loop
clean_batch_data(OUTPUT_CSV, LOGS_DIR)
df_input = pd.read_csv(INPUT_CSV)
processed = get_processed_instances(OUTPUT_CSV, LOGS_DIR)

if len(processed) < len(df_input):
    print(f"🚀 Ready to start batch Run. {len(processed)}/{len(df_input)} instances already completed.")
    print(f"🚀 Starting batch Run. {len(processed)}/{len(df_input)} instances already completed.")
else:
    print(f"🚀 {len(processed)}/{len(df_input)} instances already completed. No need further running")
    
# Modified check in Cell 4
for index, row in df_input.iterrows():
    instance_name = row['Instance']
    log_file_path = os.path.join(LOGS_DIR, f"{instance_name}_log.txt")
    
    # Check BOTH the CSV record AND the actual log file
    if instance_name in processed and os.path.exists(log_file_path):
        continue # Safe to skip
    
    optimal_cost = row['Cost']
    file_path = os.path.join(DATA_DIR, instance_name)
    print(f"\nProcessing {instance_name} (Ref Cost: {optimal_cost})...")

    try:
        # A. Update Global Data for this instance
        current_number_of_customers, current_distance_list = read_tsp_cappart_format(file_path)
        
        # B. Configure Params
        params = EAHyperparameters(
            # --- 1. Population ---
            population_size=POPULATION_SIZE,          
            generations=GENERATIONS,
            crossover_rate=CROSSOVER_RATE,
            mutation_rate=MUTATION_RATE,
            elitism_rate=ELITISM_RATE,           

            # --- 2. Ranges & Constraints ---
            lb_range_of_constant=LB_range_of_constant,
            ub_range_of_constant=UB_range_of_constant,
            min_chromosome_length=min_chromosome_length,     
            max_chromosome_length=max_chromosome_length,   

            # --- 3. Operator Specifics ---
            tournament_size=tournament_size,                             
            tournament_probability=tournament_probability,                    
            mutation_max_subtree_depth=random.randint(min_chromosome_length, max_chromosome_length),                
            homology_1_point_crossover_probability=homology_1_point_crossover_probability,    
            subtree_crossover_probability=subtree_crossover_probability,             
            uniform_crossover_probability=uniform_crossover_probability,             

            # --- 4. Problem Specific ---
            reference_point=optimal_cost,         
            solver_time_limit=SOLVER_TIME_LIMIT,
            
            # Optional: You can override available operations if needed
            available_operations=["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]
        )

        # C. Run EA (Redirecting output to file to keep notebook clean)
        log_file = os.path.join(LOGS_DIR, f"{instance_name}_log.txt")
        start_time = time.time()
        
        # Capture print outputs to log file
        with open(log_file, "w", encoding="utf-8") as f:
            with contextlib.redirect_stdout(f):
                best_ind = evolution_algorithm_execution(
                    didp_model_registry=creation_of_didp_model_function,
                    dual_bound_expression_function=dual_bound_expression_function,
                    params=params
                )
                # --- NEW CODE: Print best_ind here to save it to the log ---
                print("\n" + "="*40)
                print("FINAL BEST INDIVIDUAL")
                print("="*40)
                print(best_ind) 
                # -----------------------------------------------------------
        
        total_time = time.time() - start_time

        # D. Save Results
        result_data = {
            "Instance": instance_name,
            "Total_Time_(s)": round(total_time, 2),
            "Best_Fitness": best_ind['fitness'],
            "Best_Chromosome": str(best_ind['chromosome']),
            "Log_File": log_file
        }
        append_result_to_csv(result_data, OUTPUT_CSV)
        print(f"   ✅ Finished! Best Fit: {best_ind['fitness']:.4f} | Time: {total_time:.2f}s")

    except Exception as e:
        print(f"   ❌ Failed: {e}")
    
    finally:
        # E. Cleanup Memory
        gc.collect()

print("\n🎉 Batch Run Complete!")

🧹 Starting Data Cleanup...
   -> CSV saved. Rows: 19 (was 19).
✨ Data Cleanup Complete.

🚀 Ready to start batch Run. 19/20 instances already completed.
🚀 Starting batch Run. 19/20 instances already completed.

Processing 95.txt (Ref Cost: 371)...
   ✅ Finished! Best Fit: 0.0000 | Time: 50.80s

🎉 Batch Run Complete!
